In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [ ]:
# Load the dataset
data = pd.read_csv('../Data/churn_data.csv')  # Replace 'your_dataset_path.csv' with your actual file path

# Display the first few rows of the dataset
print(data.head())

# Check the dataset for any missing values
print(data.isnull().sum())

# Split the dataset into features (X) and target (y)
X = data.drop('churn', axis=1)
y = data['churn']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
print("\nData Types:")
print(data.dtypes)

In [ ]:
# List of categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Display unique values for each categorical column
for col in categorical_cols:
    print(f"Unique values in {col}:")
    print(X[col].unique())
    print("-------------------------------")


In [ ]:
# Display value counts for the target feature 'churn'
print(data['churn'].value_counts())


In [ ]:
# Identify categorical columns (assuming you've already identified them as categorical_cols)
categorical_cols = [col for col in X.columns if X[col].dtype == 'object']

# Create a ColumnTransformer to apply one-hot encoding to categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first'), categorical_cols)
    ],
    remainder='passthrough'
)

# Fit and transform the training data
X_train_encoded = preprocessor.fit_transform(X_train)

# Transform the testing data
X_test_encoded = preprocessor.transform(X_test)

# Convert the encoded data back to DataFrame for better understanding
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=preprocessor.get_feature_names_out())
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=preprocessor.get_feature_names_out())

# Display the first few rows of the encoded training data
print(X_train_encoded_df.head())


In [ ]:
# Identify numerical columns
numerical_cols = ['monthly_charges', 'total_charges']

# Create a ColumnTransformer to standardize numerical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first'), categorical_cols),
        ('scaler', StandardScaler(), numerical_cols)
    ],
    remainder='passthrough'
)

# Fit and transform the training data
X_train_preprocessed = preprocessor.fit_transform(X_train)

# Transform the testing data
X_test_preprocessed = preprocessor.transform(X_test)

# Convert the preprocessed data back to DataFrame for better understanding
X_train_preprocessed_df = pd.DataFrame(X_train_preprocessed, columns=preprocessor.get_feature_names_out())
X_test_preprocessed_df = pd.DataFrame(X_test_preprocessed, columns=preprocessor.get_feature_names_out())

# Display the first few rows of the preprocessed training data
print(X_train_preprocessed_df.head())


In [ ]:
# Display the first few rows of the preprocessed training data
print(X_train_preprocessed_df.head())


In [ ]:
# Display value counts for the target feature 'churn'
print(y_train.value_counts())


In [ ]:
from imblearn.combine import SMOTEENN

# Initialize SMOTEENN
sm = SMOTEENN()

# Resample the dataset
X_res, y_res = sm.fit_resample(X_train_preprocessed_df, y_train)

# Display the value counts after resampling
print(pd.Series(y_res).value_counts())



In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the target variable
y_res_encoded = label_encoder.fit_transform(y_res)

# Convert the encoded target variable back to original classes for better understanding
encoded_classes = label_encoder.classes_

# Display the encoded target variable and its classes
print("Encoded target variable:")
print(y_res_encoded)
print("\nEncoded classes:")
print(encoded_classes)



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize RandomForestClassifier
rf_classifier = RandomForestClassifier(random_state=42)

# Train the classifier on the preprocessed and balanced training data
rf_classifier.fit(X_res, y_res_encoded)

# Predict on the preprocessed testing data
y_pred = rf_classifier.predict(X_test_preprocessed_df)

# Convert the predicted values back to original classes
y_pred_original = label_encoder.inverse_transform(y_pred)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred_original)
print(f"Accuracy: {accuracy:.4f}\n")

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_original))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_original))




In [ ]:
from sklearn.model_selection import cross_val_score

# Initialize RandomForestClassifier
rf_classifier_cv = RandomForestClassifier(random_state=42)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(rf_classifier_cv, X_res, y_res_encoded, cv=5, scoring='accuracy')

# Print cross-validation scores
print("Cross-validation scores:", cv_scores)
print(f"Mean Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")


In [ ]:
from sklearn.model_selection import cross_val_predict

# Perform cross-validation predictions
y_pred_cv = cross_val_predict(rf_classifier_cv, X_res, y_res_encoded, cv=5)

# Convert the predicted values back to original classes
y_pred_cv_original = label_encoder.inverse_transform(y_pred_cv)

# Print classification report after cross-validation
print("Classification Report after Cross-Validation:")
print(classification_report(y_res, y_pred_cv_original))


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardize the data
scaler = StandardScaler()
X_res_scaled = scaler.fit_transform(X_res)

# Apply PCA
pca = PCA(n_components=0.95, random_state=42)  # Keep 95% of the variance
X_res_pca = pca.fit_transform(X_res_scaled)

# Print the number of components selected
print(f"Number of components selected: {pca.n_components_}")

# Perform cross-validation with PCA-transformed data
cv_scores_pca = cross_val_score(rf_classifier_cv, X_res_pca, y_res_encoded, cv=5, scoring='accuracy')

# Print cross-validation scores with PCA
print("Cross-validation scores with PCA:", cv_scores_pca)
print(f"Mean Accuracy with PCA: {cv_scores_pca.mean():.4f}")
print(f"Standard Deviation with PCA: {cv_scores_pca.std():.4f}")


In [ ]:
# Initialize RandomForestClassifier
rf_classifier_pca = RandomForestClassifier(random_state=42)

# Perform 5-fold cross-validation with PCA-transformed data
cv_scores_pca = cross_val_score(rf_classifier_pca, X_res_pca, y_res_encoded, cv=5, scoring='accuracy')

# Print cross-validation scores with PCA-transformed data
print("Cross-validation scores with PCA-transformed data:", cv_scores_pca)
print(f"Mean Accuracy with PCA: {cv_scores_pca.mean():.4f}")
print(f"Standard Deviation with PCA: {cv_scores_pca.std():.4f}")

# Fit the model on the entire PCA-transformed training data
rf_classifier_pca.fit(X_res_pca, y_res_encoded)


In [ ]:
import joblib

# 1. Save PCA object
joblib.dump(pca, 'Exported/pca_transformer.pkl')

# 2. Save StandardScaler object
joblib.dump(scaler, 'Exported/standard_scaler.pkl')

# 4. Save trained RandomForestClassifier model
joblib.dump(rf_classifier_pca, 'Exported/rf_classifier_pca_model.pkl')

joblib.dump(preprocessor, 'Exported/preprocessor.pkl')
# Save the list of categorical columns
joblib.dump(categorical_cols, 'Exported/categorical_cols.pkl')

In [ ]:
import joblib
# Load PCA object
pca_loaded = joblib.load('../Exported/pca_transformer.pkl')

# Load StandardScaler object
scaler_loaded = joblib.load('../Exported/standard_scaler.pkl')

# Load the saved ColumnTransformer object
preprocessor_loaded = joblib.load('../Exported/preprocessor.pkl')

# Load the list of categorical columns
categorical_cols_loaded = joblib.load('../Exported/categorical_cols.pkl')

# Load trained RandomForestClassifier model
rf_classifier_loaded = joblib.load('../Exported/rf_classifier_pca_model.pkl')



In [ ]:
import pandas as pd

# Load the new dataset
test_data = pd.read_csv('../Data/test.csv')

# Display the first few rows of the new dataset
print(test_data.head())


In [ ]:
# Identify numerical columns
numerical_cols = ['monthly_charges', 'total_charges']

# Transform the new data using the loaded ColumnTransformer
X_new_preprocessed = preprocessor_loaded.transform(test_data)

# Apply PCA transformation
X_new_pca = pca_loaded.transform(X_new_preprocessed)

# Make predictions
y_pred_new = rf_classifier_loaded.predict(X_new_pca)

# Create a DataFrame to store predictions
predictions_df = pd.DataFrame({
    'Predicted_Churn': y_pred_new
})

# Display all the predictions without truncation
pd.set_option('display.max_rows', None)  # Set the option to display all rows
print("\nAll Predictions for the new dataset:")
print(predictions_df)
